In [3]:
import pandas as pd

In [18]:

adsl = pd.read_csv("../csv_file/ADSL_PDS2019.csv").set_index("SUBJID")
adsl

,TRT,ATRT,SXANY,DTHDYX,DTHX,PFSDYCR,PFSCR,DIAGMONS,AGE,SEX,B_WEIGHT,RACE,B_ECOG,HISSUBTY,DIAGTYPE
SUBJID,,,,,,,,,,,,,,,
1,panit. plus best supportive care,panit. plus best supportive care,Y,218,1,84,1,55.0,68,Male,48.0,White or Caucasian,Symptoms but ambulatory,No sub-type,Colon
2,Best supportive care,Best supportive care,Y,231,1,57,1,25.0,48,Male,88.0,White or Caucasian,Symptoms but ambulatory,Mucinous,Colon
4,Best supportive care,Best supportive care,Y,581,1,581,1,19.0,70,Female,57.0,White or Caucasian,Fully active,No sub-type,Colon
5,Best supportive care,Best supportive care,Y,15,1,15,1,7.0,68,Female,64.4,White or Caucasian,In bed less than 50% of the time,No sub-type,Colon
6,panit. plus best supportive care,panit. plus best supportive care,Y,286,1,112,1,30.0,56,Male,83.5,White or Caucasian,Fully active,No sub-type,Rectal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
458,Best supportive care,Best supportive care,Y,501,1,54,1,36.0,71,Female,63.0,White or Caucasian,Symptoms but ambulatory,No sub-type,Rectal
459,Best supportive care,Best supportive care,Y,485,0,55,1,NaN,32,Female,62.5,White or Caucasian,Fully active,No sub-type,Rectal
460,Best supportive care,Best supportive care,Y,64,1,46,1,39.0,53,Female,75.0,White or Caucasian,Fully active,No sub-type,Colon


In [24]:
adsl['TRT'].value_counts()

TRT
Best supportive care                186
panit. plus best supportive care    184
Name: count, dtype: int64

In [25]:
# 배정받은 치료법 이랑 실제 받은 치료법이 다른 사람 (SUBJID = 283)
adsl[adsl["TRT"] != adsl["ATRT"]]

,TRT,ATRT,SXANY,DTHDYX,DTHX,PFSDYCR,PFSCR,DIAGMONS,AGE,SEX,B_WEIGHT,RACE,B_ECOG,HISSUBTY,DIAGTYPE
SUBJID,,,,,,,,,,,,,,,
283,panit. plus best supportive care,Best supportive care,Y,1,1,1,1,35.0,67,Male,55.5,White or Caucasian,In bed less than 50% of the time,No sub-type,Colon


In [30]:
# 사망한 사람 (SUBJID)
adsl[adsl["DTHX"]==0].index

Index([ 34,  37,  68,  82, 123, 126, 140, 145, 149, 181, 221, 229, 233, 258,
       261, 271, 285, 292, 300, 304, 305, 336, 343, 357, 360, 371, 372, 373,
       376, 397, 409, 437, 445, 447, 459],
      dtype='int64', name='SUBJID')

In [17]:
adls = adls.drop(columns=["Unnamed: 11", "Unnamed: 12","LSTYPE"])
adrsp = adrsp.drop(columns=["Unnamed: 8","Unnamed: 9","Unnamed: 10"])

In [10]:
adls = adls.drop(columns=["LSTYPE"])
adls

,SUBJID,VISITDY,VISIT,LSCAT,LSNEW,LSSLD,LSLD,LSSITE,LSNEWANY,LSREADER
0,1,85.0,Week 12,Target lesion,N,210.0,32.0,Liver,N,Radiologist 1
1,1,85.0,Week 12,Target lesion,N,300.0,51.0,Liver,Y,Radiologist 2
2,1,85.0,Week 12,Target lesion,N,210.0,53.0,Liver,N,Radiologist 1
3,1,85.0,Week 12,Target lesion,N,300.0,37.0,Liver,Y,Radiologist 2
4,1,85.0,Week 12,Target lesion,N,210.0,32.0,Liver,N,Radiologist 1
...,...,...,...,...,...,...,...,...,...,...
9930,461,-1.0,Screening,Non-target lesion,N,245.0,NaN,Pulmonary,N,Radiologist 2
9931,461,-1.0,Screening,Non-target lesion,N,219.0,NaN,Lymph Nodes,N,Radiologist 1
9932,461,-1.0,Screening,Non-target lesion,N,245.0,NaN,Lymph Nodes,N,Radiologist 2
9933,461,-1.0,Screening,Non-target lesion,N,219.0,NaN,Gastrointestinal,N,Radiologist 1


In [24]:
adrsp = adrsp[(adrsp['RSRESP'] != 'Unknown') & (adrsp['VISIT'] != 'Screening')].copy()

In [21]:
adrsp['RSRESP'].value_counts()

RSRESP
Progressive disease    833
Stable disease         276
Partial response       145
Unable to evaluate     100
Name: count, dtype: int64

In [39]:
adsl = pd.read_csv("csv_file/ADSL_PDS2019.csv")
adrsp = pd.read_csv("csv_file/ADRSP_PDS2019.csv")

adrsp = adrsp[(adrsp['RSRESP'] != 'Unknown') & (adrsp['VISIT'] != 'Screening')].copy()

# Radiologist 1 으로 기준
reader_rank = {'Radiologist 1': 1,'Radiologist 2':2,'Oncologist':3}
adrsp['_rank'] = adrsp['RSREADER'].map(reader_rank)

adrsp = (adrsp.sort_values(by=['SUBJID','VISITDY','_rank'])
         .groupby(['SUBJID', 'VISITDY'], as_index=False)
          .first())
resp_rank = {
    "Complete response":   1, # 없지만 그냥 넣어놈(국제 표준이라고해서)
    "Partial response":    2,
    "Stable disease":      3,
    "Progressive disease": 4,
    "Unable to evaluate":  5,
}
adrsp['_resp'] = adrsp['RSRESP'].map(resp_rank)

# BOR로 가장 좋았던 판정으로 판정
adrsp = (adrsp.sort_values(['SUBJID', '_resp'])
            .groupby('SUBJID', as_index=False)
            .first()[['SUBJID', 'RSRESP']]
            .rename(columns={'RSRESP': 'BOR'}))

adrsp['RESPONDER'] = adrsp['BOR'].isin(['Complete response', 'Partial response']).astype(int)

final = adsl.merge(adrsp, on='SUBJID', how='left')

final['BOR'] = final['BOR'].fillna('unknown')

# 43명이 평가를 못 받아서 0으로
final['RESPONDER'] = final['RESPONDER'].fillna(0).astype(int)


In [42]:
final.to_csv("csv_file/adsl_responder.csv", index=False)

In [28]:
# Radiologist 1 으로 기준
adrsp1 = adrsp[(adrsp['RSRESP'] != 'Unknown') & (adrsp['VISIT'] != 'Screening')].copy()
reader_rank = {'Radiologist 1': 2,'Radiologist 2':1,'Oncologist':1}
adrsp1['_rank'] = adrsp['RSREADER'].map(reader_rank)

adrsp1 = (adrsp.sort_values(by=['SUBJID','VISITDY','_rank'])
         .groupby(['SUBJID', 'VISITDY'], as_index=False)
          .first())

adrsp1['_rank'].value_counts()

_rank
1    368
3    250
2      6
Name: count, dtype: int64

In [15]:
bio = pd.read_csv('../csv_file/BIOMARK_PDS2019.csv')
bio

,SUBJID,BMMTNM1,BMMTR1,BMMTNM2,BMMTR2,BMMTNM3,BMMTR3,BMMTNM4,BMMTR4,BMMTNM5,BMMTR5,BMMTNM6,BMMTR6,BMMTNM7,BMMTR7
0,1,KRAS exon 2 (c12/13),Failure,KRAS exon 3 (c61),NaN,KRAS exon 4 (c117/146),NaN,NRAS exon 2 (c12/13),NaN,NRAS exon 3 (c61),NaN,NRAS exon 4 (c117/146),NaN,BRAF exon 15 (c600),NaN
1,2,KRAS exon 2 (c12/13),Wild-type,KRAS exon 3 (c61),NaN,KRAS exon 4 (c117/146),NaN,NRAS exon 2 (c12/13),NaN,NRAS exon 3 (c61),NaN,NRAS exon 4 (c117/146),NaN,BRAF exon 15 (c600),NaN
2,4,KRAS exon 2 (c12/13),Mutant,KRAS exon 3 (c61),NaN,KRAS exon 4 (c117/146),NaN,NRAS exon 2 (c12/13),NaN,NRAS exon 3 (c61),NaN,NRAS exon 4 (c117/146),NaN,BRAF exon 15 (c600),NaN
3,5,KRAS exon 2 (c12/13),Mutant,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Wild-type,NRAS exon 2 (c12/13),Wild-type,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Wild-type,BRAF exon 15 (c600),Wild-type
4,6,KRAS exon 2 (c12/13),Wild-type,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Wild-type,NRAS exon 2 (c12/13),Wild-type,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Wild-type,BRAF exon 15 (c600),Wild-type
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389,458,KRAS exon 2 (c12/13),Wild-type,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Wild-type,NRAS exon 2 (c12/13),Wild-type,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Wild-type,BRAF exon 15 (c600),Wild-type
390,459,KRAS exon 2 (c12/13),Wild-type,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Wild-type,NRAS exon 2 (c12/13),Wild-type,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Wild-type,BRAF exon 15 (c600),Wild-type
391,460,KRAS exon 2 (c12/13),Failure,KRAS exon 3 (c61),NaN,KRAS exon 4 (c117/146),NaN,NRAS exon 2 (c12/13),NaN,NRAS exon 3 (c61),NaN,NRAS exon 4 (c117/146),NaN,BRAF exon 15 (c600),NaN
392,461,KRAS exon 2 (c12/13),Mutant,KRAS exon 3 (c61),NaN,KRAS exon 4 (c117/146),NaN,NRAS exon 2 (c12/13),NaN,NRAS exon 3 (c61),NaN,NRAS exon 4 (c117/146),NaN,BRAF exon 15 (c600),NaN


In [47]:
final = pd.read_csv('../csv_file/adsl_responder.csv')
final['RESPONDER'].value_counts()

RESPONDER
0    345
1     25
Name: count, dtype: int64

In [18]:
#biomark에서 subjID가 중복값 ( 24 )
dup = bio['SUBJID'].value_counts()
dup = dup[dup > 1]
print(len(dup))

bio[bio['SUBJID'].isin(dup.index)].sort_values('SUBJID')

24


,SUBJID,BMMTNM1,BMMTR1,BMMTNM2,BMMTR2,BMMTNM3,BMMTR3,BMMTNM4,BMMTR4,BMMTNM5,BMMTR5,BMMTNM6,BMMTR6,BMMTNM7,BMMTR7
42,55,KRAS exon 2 (c12/13),Mutant,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Wild-type,NRAS exon 2 (c12/13),Wild-type,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Failure,BRAF exon 15 (c600),Failure
43,55,KRAS exon 2 (c12/13),Mutant,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Wild-type,NRAS exon 2 (c12/13),Wild-type,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Wild-type,BRAF exon 15 (c600),Failure
48,60,KRAS exon 2 (c12/13),Wild-type,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Failure,NRAS exon 2 (c12/13),Wild-type,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Failure,BRAF exon 15 (c600),Wild-type
49,60,KRAS exon 2 (c12/13),Wild-type,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Wild-type,NRAS exon 2 (c12/13),Wild-type,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Wild-type,BRAF exon 15 (c600),Wild-type
82,97,KRAS exon 2 (c12/13),Wild-type,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Failure,NRAS exon 2 (c12/13),Mutant,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Failure,BRAF exon 15 (c600),Wild-type
83,97,KRAS exon 2 (c12/13),Wild-type,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Wild-type,NRAS exon 2 (c12/13),Mutant,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Wild-type,BRAF exon 15 (c600),Wild-type
109,136,KRAS exon 2 (c12/13),Mutant,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Failure,NRAS exon 2 (c12/13),Wild-type,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Wild-type,BRAF exon 15 (c600),Wild-type
110,136,KRAS exon 2 (c12/13),Mutant,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Wild-type,NRAS exon 2 (c12/13),Wild-type,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Wild-type,BRAF exon 15 (c600),Wild-type
112,140,KRAS exon 2 (c12/13),Wild-type,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Failure,NRAS exon 2 (c12/13),Wild-type,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Wild-type,BRAF exon 15 (c600),Wild-type
113,140,KRAS exon 2 (c12/13),Wild-type,KRAS exon 3 (c61),Wild-type,KRAS exon 4 (c117/146),Wild-type,NRAS exon 2 (c12/13),Wild-type,NRAS exon 3 (c61),Wild-type,NRAS exon 4 (c117/146),Wild-type,BRAF exon 15 (c600),Wild-type


In [48]:
# 재검사 사람 있어서 마지막 결과를 사용해서 24개 중복값을 처리
bio_clean = bio.groupby('SUBJID', as_index=False).last()



In [49]:
final = final.merge(
    bio_clean[['SUBJID','BMMTR1','BMMTR2','BMMTR3','BMMTR4','BMMTR5','BMMTR6']],
    on='SUBJID',how='left'
)


In [40]:
final_kras = final[final['BMMTR1'] != 'Failure'].copy()
final_kras['Kras_bin'] = (final_kras['BMMTR1'] == 'Wild-type').astype(int)
final_kras['TRT_bin'] = (final_kras['TRT'].str.contains('panit')).astype(int)

final_kras

,SUBJID,TRT,ATRT,SXANY,DTHDYX,DTHX,PFSDYCR,PFSCR,DIAGMONS,AGE,...,BMMTR5_x,BMMTR6_x,BMMTR1,BMMTNM2,BMMTR3,BMMTR4_y,BMMTR5_y,BMMTR6_y,Kras_bin,TRT_bin
1,2,Best supportive care,Best supportive care,Y,231,1,57,1,25.0,48,...,None,None,Wild-type,KRAS exon 3 (c61),None,None,None,None,1,0
2,4,Best supportive care,Best supportive care,Y,581,1,581,1,19.0,70,...,None,None,Mutant,KRAS exon 3 (c61),None,None,None,None,0,0
3,5,Best supportive care,Best supportive care,Y,15,1,15,1,7.0,68,...,Wild-type,Wild-type,Mutant,KRAS exon 3 (c61),Wild-type,Wild-type,Wild-type,Wild-type,0,0
4,6,panit. plus best supportive care,panit. plus best supportive care,Y,286,1,112,1,30.0,56,...,Wild-type,Wild-type,Wild-type,KRAS exon 3 (c61),Wild-type,Wild-type,Wild-type,Wild-type,1,1
5,8,Best supportive care,Best supportive care,Y,43,1,35,1,39.0,66,...,None,None,Wild-type,KRAS exon 3 (c61),None,None,None,None,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
364,457,Best supportive care,Best supportive care,Y,94,1,50,1,18.0,74,...,Wild-type,Wild-type,Mutant,KRAS exon 3 (c61),Wild-type,Wild-type,Wild-type,Wild-type,0,0
365,458,Best supportive care,Best supportive care,Y,501,1,54,1,36.0,71,...,Wild-type,Wild-type,Wild-type,KRAS exon 3 (c61),Wild-type,Wild-type,Wild-type,Wild-type,1,0
366,459,Best supportive care,Best supportive care,Y,485,0,55,1,NaN,32,...,Wild-type,Wild-type,Wild-type,KRAS exon 3 (c61),Wild-type,Wild-type,Wild-type,Wild-type,1,0
368,461,panit. plus best supportive care,panit. plus best supportive care,N,175,1,53,1,9.0,76,...,None,None,Mutant,KRAS exon 3 (c61),None,None,None,None,0,1


In [45]:
final

,SUBJID,TRT,ATRT,SXANY,DTHDYX,DTHX,PFSDYCR,PFSCR,DIAGMONS,AGE,...,BMMTR3_y,BMMTR4_x,BMMTR5_x,BMMTR6_x,BMMTR1,BMMTNM2,BMMTR3,BMMTR4_y,BMMTR5_y,BMMTR6_y
0,1,panit. plus best supportive care,panit. plus best supportive care,Y,218,1,84,1,55.0,68,...,None,None,None,None,Failure,KRAS exon 3 (c61),None,None,None,None
1,2,Best supportive care,Best supportive care,Y,231,1,57,1,25.0,48,...,None,None,None,None,Wild-type,KRAS exon 3 (c61),None,None,None,None
2,4,Best supportive care,Best supportive care,Y,581,1,581,1,19.0,70,...,None,None,None,None,Mutant,KRAS exon 3 (c61),None,None,None,None
3,5,Best supportive care,Best supportive care,Y,15,1,15,1,7.0,68,...,Wild-type,Wild-type,Wild-type,Wild-type,Mutant,KRAS exon 3 (c61),Wild-type,Wild-type,Wild-type,Wild-type
4,6,panit. plus best supportive care,panit. plus best supportive care,Y,286,1,112,1,30.0,56,...,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,KRAS exon 3 (c61),Wild-type,Wild-type,Wild-type,Wild-type
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365,458,Best supportive care,Best supportive care,Y,501,1,54,1,36.0,71,...,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,KRAS exon 3 (c61),Wild-type,Wild-type,Wild-type,Wild-type
366,459,Best supportive care,Best supportive care,Y,485,0,55,1,NaN,32,...,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,KRAS exon 3 (c61),Wild-type,Wild-type,Wild-type,Wild-type
367,460,Best supportive care,Best supportive care,Y,64,1,46,1,39.0,53,...,None,None,None,None,Failure,KRAS exon 3 (c61),None,None,None,None
368,461,panit. plus best supportive care,panit. plus best supportive care,N,175,1,53,1,9.0,76,...,None,None,None,None,Mutant,KRAS exon 3 (c61),None,None,None,None


In [51]:
bm_ras = ['BMMTR1','BMMTR2','BMMTR3','BMMTR4','BMMTR5','BMMTR6']

has_failure = (final[bm_ras] == 'Failure').any(axis=1)
has_mutant = (final[bm_ras] == 'Mutant').any(axis=1)

final['RAS_status'] = 'Wild-type'
final.loc[has_mutant,  'RAS_status'] = 'Mutant'
final.loc[has_failure, 'RAS_status'] = None

final_ras = final[final['RAS_status'].notna()].copy()
final_ras['ras_bin'] = (final['RAS_status'] == 'Wild-type').astype(int)
final_ras['TRT_bin'] = (final['TRT'].str.contains('panit')).astype(int)
final_ras

,SUBJID,TRT,ATRT,SXANY,DTHDYX,DTHX,PFSDYCR,PFSCR,DIAGMONS,AGE,...,RESPONDER,BMMTR1,BMMTR2,BMMTR3,BMMTR4,BMMTR5,BMMTR6,RAS_status,ras_bin,TRT_bin
1,2,Best supportive care,Best supportive care,Y,231,1,57,1,25.0,48,...,0,Wild-type,None,None,None,None,None,Wild-type,1,0
2,4,Best supportive care,Best supportive care,Y,581,1,581,1,19.0,70,...,0,Mutant,None,None,None,None,None,Mutant,0,0
3,5,Best supportive care,Best supportive care,Y,15,1,15,1,7.0,68,...,0,Mutant,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,Mutant,0,0
4,6,panit. plus best supportive care,panit. plus best supportive care,Y,286,1,112,1,30.0,56,...,0,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,1,1
5,8,Best supportive care,Best supportive care,Y,43,1,35,1,39.0,66,...,0,Wild-type,None,None,None,None,None,Wild-type,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
364,457,Best supportive care,Best supportive care,Y,94,1,50,1,18.0,74,...,0,Mutant,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,Mutant,0,0
365,458,Best supportive care,Best supportive care,Y,501,1,54,1,36.0,71,...,0,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,1,0
366,459,Best supportive care,Best supportive care,Y,485,0,55,1,NaN,32,...,0,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,Wild-type,1,0
368,461,panit. plus best supportive care,panit. plus best supportive care,N,175,1,53,1,9.0,76,...,0,Mutant,None,None,None,None,None,Mutant,0,1


In [53]:
final_ras[['SUBJID','RAS_status','ras_bin','TRT','TRT_bin']]

,SUBJID,RAS_status,ras_bin,TRT,TRT_bin
1,2,Wild-type,1,Best supportive care,0
2,4,Mutant,0,Best supportive care,0
3,5,Mutant,0,Best supportive care,0
4,6,Wild-type,1,panit. plus best supportive care,1
5,8,Wild-type,1,Best supportive care,0
...,...,...,...,...,...
364,457,Mutant,0,Best supportive care,0
365,458,Wild-type,1,Best supportive care,0
366,459,Wild-type,1,Best supportive care,0
368,461,Mutant,0,panit. plus best supportive care,1


In [54]:
len(final_ras)

328

In [56]:
kras=pd.read_csv('../csv_file/final_ras.csv')
kras[['SUBJID','BMMTR1','Kras_bin','TRT','TRT_bin']]

,SUBJID,BMMTR1,Kras_bin,TRT,TRT_bin
0,2,Wild-type,1,Best supportive care,0
1,4,Mutant,0,Best supportive care,0
2,5,Mutant,0,Best supportive care,0
3,6,Wild-type,1,panit. plus best supportive care,1
4,8,Wild-type,1,Best supportive care,0
...,...,...,...,...,...
323,457,Mutant,0,Best supportive care,0
324,458,Wild-type,1,Best supportive care,0
325,459,Wild-type,1,Best supportive care,0
326,461,Mutant,0,panit. plus best supportive care,1


In [64]:
kras=pd.read_csv('../csv_file/final_kras.csv')
ras = pd.read_csv('../csv_file/final_ras.csv')

print(kras['BMMTR1'].value_counts())
print(kras['KRAS_bin'].value_counts())
print(pd.crosstab(kras['TRT_bin'], kras['KRAS_bin']))

print(ras['RAS_status'].value_counts())
print(ras['RAS_bin'].value_counts())
print(pd.crosstab(ras['TRT_bin'], ras['RAS_bin']))


BMMTR1
Wild-type    195
Mutant       146
Name: count, dtype: int64
KRAS_bin
1    195
0    146
Name: count, dtype: int64
KRAS_bin   0   1
TRT_bin         
0         77  99
1         69  96
RAS_status
Wild-type    167
Mutant       161
Name: count, dtype: int64
RAS_bin
1    167
0    161
Name: count, dtype: int64
RAS_bin   0   1
TRT_bin        
0        80  85
1        81  82
